# Module 06 — Lesson 05
# Feature Scaling (Normalization, Standardization)

---

> Every column is finally numeric after Lesson 04. But `age` ranges from 22 to 57, `years_experience` from 0 to 22, and `salary` from 47,100 to 245,000. To a distance-based algorithm — the kind you'll meet starting in Module 13 — a $1,000 difference in salary and a 1-year difference in experience look identical in scale, even though they mean completely different things. Feature scaling puts every column on comparable footing. Done carelessly, it can also erase the very outlier you deliberately chose to keep in Lesson 03.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Explain why unscaled features distort distance-based and gradient-based algorithms
- Apply **Min-Max normalization** to rescale a column into `[0, 1]`
- Apply **standardization (Z-score)** to rescale a column to mean 0, standard deviation 1
- Apply **robust scaling** (median/IQR) and explain why it handles outliers better than the other two
- Recognize which columns should usually be left unscaled (like one-hot dummy columns)
- Avoid fitting scaling parameters on data that includes what should be a held-out test set

In [1]:
import pandas as pd

employees = pd.read_csv("data/employees_clean.csv")
print(f"employees.shape: {employees.shape}")
employees[["employee_id", "age", "years_experience", "salary"]].describe()

employees.shape: (49, 10)


,employee_id,age,years_experience,salary
count,49.000000,49.000000,49.000000,49.000000
mean,1030.061224,38.183673,4.502041,73361.224490
std,18.201887,9.801517,4.358253,28040.237579
min,1001.000000,22.000000,0.000000,47100.000000
25%,1015.000000,28.000000,1.400000,58000.000000
50%,1028.000000,38.000000,3.100000,71800.000000
75%,1046.000000,45.000000,5.700000,79800.000000
max,1060.000000,57.000000,22.000000,245000.000000


---
## 1. Why Scale at All?

Picture a nearest-neighbor algorithm (Module 13) comparing two employees using `age`, `years_experience`, and `salary` together. A $30,000 salary gap and a 2-year experience gap would both get folded into one "distance" number — but because salary is measured in the tens of thousands and experience in single digits, the salary gap alone would dominate that distance almost completely, regardless of which difference actually matters more. Scaling puts every column on the same footing before that kind of comparison happens.

---
## 2. Min-Max Normalization

Rescales every value into `[0, 1]`: `x_scaled = (x - min) / (max - min)`. The smallest value in the column becomes 0, the largest becomes 1, everything else lands proportionally between.

In [2]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

salary_minmax = min_max_scale(employees["salary"])
salary_minmax.describe()

count    49.000000
mean      0.132699
std       0.141689
min       0.000000
25%       0.055078
50%       0.124811
75%       0.165235
max       1.000000
Name: salary, dtype: float64

Look closely at that description: the 75th percentile scaled value is only about **0.17** — meaning 75% of employees are squeezed into the bottom sixth of the `[0, 1]` range. That's the 245,000 real outlier from Lesson 03 doing damage: because Min-Max uses the raw `min` and `max`, a single extreme value stretches the scale so far that every "normal" salary gets crushed together near 0.

In [3]:
comparison = pd.DataFrame({
    "name": employees["name"],
    "salary": employees["salary"],
    "salary_minmax": salary_minmax.round(3),
}).sort_values("salary").reset_index(drop=True)

lowest_paid = comparison.iloc[0]
near_75th = comparison.iloc[(comparison["salary"] - employees["salary"].quantile(0.75)).abs().idxmin()]
real_gap = near_75th["salary"] - lowest_paid["salary"]
scaled_gap = near_75th["salary_minmax"] - lowest_paid["salary_minmax"]

print(f"{lowest_paid['name']} ({lowest_paid['salary']:,.0f}) vs. {near_75th['name']} "
      f"({near_75th['salary']:,.0f}): a real gap of {real_gap:,.0f} dollars")
print(f"...but only a {scaled_gap:.3f} gap on the Min-Max [0, 1] scale.")
comparison.iloc[[0, comparison.index[comparison['name'] == near_75th['name']][0]]]

Yash Verma (47,100) vs. Aditya Pillai (79,800): a real gap of 32,700 dollars
...but only a 0.165 gap on the Min-Max [0, 1] scale.


,name,salary,salary_minmax
0,Yash Verma,47100.0,0.000
36,Aditya Pillai,79800.0,0.165


---
## 3. Standardization (Z-Score)

Rescales every value to how many standard deviations it sits from the mean: `x_scaled = (x - mean) / std`. The result has mean 0 and standard deviation 1 — no fixed upper/lower bound, so a genuine outlier stays visibly far from the rest instead of dragging everyone else toward it.

In [4]:
def standardize(series):
    return (series - series.mean()) / series.std()

salary_standardized = standardize(employees["salary"])
salary_standardized.describe()

count    4.900000e+01
mean     1.373618e-17
std      1.000000e+00
min     -9.365550e-01
25%     -5.478279e-01
50%     -5.567801e-02
75%      2.296263e-01
max      6.121160e+00
Name: salary, dtype: float64

---
## 4. Robust Scaling — Built for Data Like This One

Standardization is *better* than Min-Max here, but `mean` and `std` still shift when an outlier is present (you saw exactly this masking effect in Lesson 03). **Robust scaling** replaces both with the median and the IQR — statistics that barely move no matter how extreme one value is: `x_scaled = (x - median) / IQR`.

In [5]:
def robust_scale(series):
    median = series.median()
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return (series - median) / iqr

salary_robust = robust_scale(employees["salary"])
salary_robust.describe()

count    49.000000
mean      0.071616
std       1.286249
min      -1.133028
25%      -0.633028
50%       0.000000
75%       0.366972
max       7.944954
Name: salary, dtype: float64

In [6]:
# Compare the middle 50% of employees (Q1-Q3) under each method -- this is where
# most of your data actually lives, and where you most need the scaling to preserve differences
middle_50pct_range = pd.DataFrame({
    "method": ["Min-Max", "Standardization", "Robust"],
    "Q1_scaled": [salary_minmax.quantile(0.25), salary_standardized.quantile(0.25), salary_robust.quantile(0.25)],
    "Q3_scaled": [salary_minmax.quantile(0.75), salary_standardized.quantile(0.75), salary_robust.quantile(0.75)],
})
middle_50pct_range["spread"] = middle_50pct_range["Q3_scaled"] - middle_50pct_range["Q1_scaled"]
middle_50pct_range

,method,Q1_scaled,Q3_scaled,spread
0,Min-Max,0.055078,0.165235,0.110157
1,Standardization,-0.547828,0.229626,0.777454
2,Robust,-0.633028,0.366972,1.000000


Min-Max squeezes the middle 50% of employees into a spread of about **0.11** out of a total possible range of 1.0 -- barely any room to tell typical employees apart. Robust scaling gives that same middle 50% a spread around **1.0**, preserving real differences, while still letting the genuine 245,000 outlier land far out at the extreme (over 7) where it honestly belongs.

| Method | Formula | Sensitive to outliers? | Best for |
|---|---|---|---|
| Min-Max | `(x - min) / (max - min)` | Very (uses the most extreme values directly) | Bounded input needed (e.g. neural nets), and no major outliers |
| Standardization | `(x - mean) / std` | Somewhat (mean/std shift with outliers) | Algorithms assuming roughly Gaussian data (linear/logistic regression, PCA) |
| Robust | `(x - median) / IQR` | Least (median/IQR barely move) | Data with real outliers you want to keep, like this one |

---
## 5. One Column You Usually Don't Scale

The `dept_*` one-hot columns from Lesson 04 are already `0`/`1`. Scaling them mathematically "works" but adds nothing — a binary column has no meaningful notion of standard deviation beyond its two possible values, and scaling it can actually make the model harder to interpret. The columns that need scaling are the continuous ones: `age`, `years_experience`, `salary`, and the ordinal `performance_band_encoded`.

In [7]:
columns_to_scale = ["age", "years_experience", "salary", "performance_band_encoded"]
columns_left_alone = [c for c in employees.columns if c.startswith("dept_")]

print(f"Scale these: {columns_to_scale}")
print(f"Leave these alone: {columns_left_alone}")

Scale these: ['age', 'years_experience', 'salary', 'performance_band_encoded']
Leave these alone: ['dept_Finance', 'dept_HR', 'dept_Marketing', 'dept_Sales']


---
## ⚠️ Common Mistakes

In [8]:
# -- Mistake 1: Using Min-Max by default without checking for outliers first ----------------

print("Mistake 1 — reaching for Min-Max scaling as the 'default' choice compresses the middle")
print("50% of salaries into a spread of just:")
print(f"  {(salary_minmax.quantile(0.75) - salary_minmax.quantile(0.25)):.3f} out of a possible 1.0 range")
print("  -> Check for outliers (Lesson 03's tools) before choosing Min-Max over Robust scaling.")

Mistake 1 — reaching for Min-Max scaling as the 'default' choice compresses the middle
50% of salaries into a spread of just:
  0.110 out of a possible 1.0 range
  -> Check for outliers (Lesson 03's tools) before choosing Min-Max over Robust scaling.


In [9]:
# -- Mistake 2: Computing scaling parameters (min/max, mean/std) on the FULL dataset, --------
# --             including rows that should have been held out as a test set ----------------

train_demo = employees.sample(frac=0.8, random_state=1)
test_demo = employees.drop(train_demo.index)

wrong_min, wrong_max = employees["salary"].min(), employees["salary"].max()  # uses ALL rows
right_min, right_max = train_demo["salary"].min(), train_demo["salary"].max()  # train rows only

print("Mistake 2 — scaling parameters learned from the whole dataset leak test-set information")
print("into training, the same leakage bug from Lesson 01's imputation section:")
print(f"  min/max from ALL rows (wrong):    {wrong_min:,.0f} / {wrong_max:,.0f}")
print(f"  min/max from train rows only:     {right_min:,.0f} / {right_max:,.0f}")
print("  -> Split first, fit scaling parameters on train only, then apply them to test.")

Mistake 2 — scaling parameters learned from the whole dataset leak test-set information
into training, the same leakage bug from Lesson 01's imputation section:
  min/max from ALL rows (wrong):    47,100 / 245,000
  min/max from train rows only:     50,700 / 245,000
  -> Split first, fit scaling parameters on train only, then apply them to test.


In [10]:
# -- Mistake 3: Scaling the one-hot dummy columns along with the continuous ones -------------

dept_col = "dept_Engineering" if "dept_Engineering" in employees.columns else columns_left_alone[0]
accidentally_scaled = standardize(employees[dept_col].astype(float))

print(f"Mistake 3 — standardizing a 0/1 column like '{dept_col}' turns clean binary")
print("membership into odd fractional values that are harder to read and add no information:")
print(f"  Original values: {sorted(employees[dept_col].unique())}")
print(f"  'Standardized' values: {sorted(accidentally_scaled.unique().round(3))}")
print("  -> Leave one-hot columns as 0/1; only scale genuinely continuous columns.")

Mistake 3 — standardizing a 0/1 column like 'dept_Finance' turns clean binary
membership into odd fractional values that are harder to read and add no information:
  Original values: [np.False_, np.True_]
  'Standardized' values: [np.float64(-0.37), np.float64(2.65)]
  -> Leave one-hot columns as 0/1; only scale genuinely continuous columns.


---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Min-Max Scale `age`

Load `data/employees_clean.csv` fresh. Apply `min_max_scale()` to `age`. Print the min, max, and the 25th/75th percentile of the result.

In [11]:
# Your code here

### Exercise 2 — Standardize `years_experience`

Apply `standardize()` to `years_experience`. Confirm the resulting mean is (approximately) 0 and the standard deviation is (approximately) 1.

In [12]:
# Your code here

### Exercise 3 — Compare All Three on `age`

Apply Min-Max, standardization, and robust scaling to `age` (which has no extreme outliers, unlike `salary`). Compare the middle-50% spread for all three, the way Section 4 did for `salary`. Are the differences between methods as dramatic here? Explain in a comment why or why not.

In [13]:
# Your code here

### Exercise 4 — Scale Only What Should Be Scaled

Build a new DataFrame `employees_scaled` where `age`, `years_experience`, `salary`, and `performance_band_encoded` are robust-scaled, but `employee_id`, `name`, and every `dept_*` column are left untouched.

In [14]:
# Your code here

### Exercise 5 — (Challenge) Simulate the Leakage Bug

Split `employees` into an 80/20 train/test set (as in the Mistake 2 demo). Compute robust-scaled `salary` two ways: (a) median/IQR from the full dataset, (b) median/IQR from the training rows only, applied to both train and test. Print both sets of scaling parameters and comment on how different they are for this dataset.

In [15]:
# Your code here

---
## 📌 Key Takeaways

- **Unscaled features let whichever column has the biggest raw numbers dominate distance- and gradient-based algorithms**, regardless of which feature actually matters more.
- **Min-Max normalization is the most outlier-sensitive scaler** — a single extreme value stretches `[min, max]` so far that every typical value gets crushed together. Prefer robust scaling (median/IQR) when real outliers are present and worth keeping.
- **Fit scaling parameters on training data only, and don't scale columns that are already 0/1** (like one-hot dummies) — both mistakes either leak test information into training or add noise without adding information.